[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/runnerup96/SBT-Advanced-ML/blob/main/information%20retrieval/Learning%20To%20Rank.ipynb)

### Learning To Rank

Learning to Rank (LTR) — область задач машинного обучения, решающая задачу упорядочивания объектов (документов, товаров, объявлений) по релевантности к запросу пользователя. В отличие от классификации или регрессии, LTR оптимизирует относительный порядок элементов, а не их абсолютные значения.

Мы используем датасет Yahoo! Learning to Rank в стандартном LETOR-формате:

| Relevance | qid | doc_id | Feature Vector |
| :--- | :--- | :--- | :--- |
| 2 | 123 | 456 | 1:0.85 2:0.03 ... 699:0.12 |
| 0 | 123 | 789 | 1:0.21 2:0.91 ... 699:0.04 |
| 3 | 456 | 112 | 1:0.77 2:0.44 ... 699:0.88 |


Каждая строка = вектор пары «запрос-документ»:
* label = градация релевантности (0–4)
* qid = идентификатор запроса (группирует документы одного запроса)
* 699 признаков в разреженном формате → для эффективности отбираем 100 самых ненулевых признаков

Скачайте файл для обучения отсюда: [LETOR Yahoo Dataset](https://drive.google.com/file/d/19KviAMIu-MvlfOm_frbe6PLgkUvHpVnh/view?usp=sharing)

Все файлы Yahoo LETOR -- https://huggingface.co/datasets/YahooResearch/Yahoo-Learning-to-Rank-Challenge

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import ndcg_score
import matplotlib.pyplot as plt

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use device 0

In [ ]:
def parse_letor_line(line):
    """Парсинг строк датасета LETOR"""
    tokens = line.strip().split()
    if not tokens or tokens[0].startswith('#'):
        return None  

    label = int(float(tokens[0]))  

    # 2. Query ID
    qid_token = tokens[1]
    assert qid_token.startswith('qid:')
    qid = int(qid_token.split(':')[1])

    features = {}
    comment = None
    for token in tokens[2:]:
        if token.startswith('#'):
            comment = ' '.join(tokens[tokens.index(token):])[1:].strip()
            break
        if ':' in token:
            idx, val = token.split(':')
            features[int(idx)] = float(val)

    return {
        'label': label,
        'qid': qid,
        'features': features,  
        'comment': comment
    }

Прочитаем и подготовим наши данные для обучения задачи ранжирования.

In [ ]:
def load_letor_dataset(filepath, num_features=700, dense=True):
    """
    Загрузка LETOR датасета
    """
    labels, qids, features_list = [], [], []

    with open(filepath) as f:
        for line in f:
            parsed = parse_letor_line(line)
            if parsed is None:
                continue
            labels.append(parsed['label'])
            qids.append(parsed['qid'])
            if dense:
                vec = np.zeros(num_features, dtype=np.float32)
                for idx, val in parsed['features'].items():
                    vec[idx - 1] = val  
                features_list.append(vec)
            else:
                features_list.append(parsed['features'])

    labels = np.array(labels)
    qids = np.array(qids)
    X = np.vstack(features_list) if dense else features_list

    return labels, qids, X

def prepare_letor_splits(filepath, train_queries=1000, test_queries=266, top_k_features=100, random_state=42):
    """
    Подготовка обучающую и тестовую выборки LETOR с разделением по запросам и отбором признаков.

    Args:
        filepath: Путь к файлу с набором данных LETOR
        train_queries: Количество уникальных запросов для обучения (по умолчанию: 1000)
        test_queries: Количество уникальных запросов для тестирования (по умолчанию: 266)
        top_k_features: Количество наименее разреженных признаков для сохранения (по умолчанию: 100)
        random_state: Случайный seed для воспроизводимого разделения запросов
    
    Returns:
        df_train: DataFrame со столбцами ['qid', 'label', 'f1', ..., 'f100']
        df_test: DataFrame того же формата для тестовой выборки
        selected_features: Массив исходных индексов признаков LETOR (нумерация с 1), соответствующих f1..f100
    """
    labels, qids, X_dense = load_letor_dataset(filepath, num_features=700)
    
    unique_qids = np.unique(qids)
    total_needed = train_queries + test_queries
    if len(unique_qids) < total_needed:
        raise ValueError(
            f"Dataset contains only {len(unique_qids)} unique queries, "
            f"but {total_needed} are required ({train_queries} train + {test_queries} test)"
        )
    
    rng = np.random.RandomState(random_state)
    shuffled_qids = rng.permutation(unique_qids)
    train_qids = shuffled_qids[:train_queries]
    test_qids = shuffled_qids[train_queries:total_needed]
    
    non_zero_counts = np.sum(X_dense != 0, axis=0)  # Shape: (699,)
    
    # Select top-K least sparse features (highest non-zero counts)
    top_indices = np.argsort(non_zero_counts)[::-1][:top_k_features]  # 0-based internal indices
    selected_features = top_indices + 1  # Convert to 1-based LETOR feature numbers
    
    # Create query masks
    train_mask = np.isin(qids, train_qids)
    test_mask = np.isin(qids, test_qids)
    
    # Extract subsets with selected features only
    X_train = X_dense[train_mask][:, top_indices]
    y_train = labels[train_mask]
    q_train = qids[train_mask]
    
    X_test = X_dense[test_mask][:, top_indices]
    y_test = labels[test_mask]
    q_test = qids[test_mask]
    
    def _to_dataframe(X, y, qids_arr):
        n_samples = X.shape[0]
        data = []
        for i in range(n_samples):
            row = {'qid': int(qids_arr[i]), 'label': int(y[i]) if float(y[i]).is_integer() else y[i]}
            for j in range(top_k_features):
                row[f'f{j+1}'] = X[i, j]
            data.append(row)
        return pd.DataFrame(data)
    
    df_train = _to_dataframe(X_train, y_train, q_train)
    df_test = _to_dataframe(X_test, y_test, q_test)
    
    expected_cols = ['qid', 'label'] + [f'f{i}' for i in range(1, top_k_features + 1)]
    assert list(df_train.columns) == expected_cols, "Train DataFrame has incorrect columns"
    assert list(df_test.columns) == expected_cols, "Test DataFrame has incorrect columns"
    
    return df_train, df_test, selected_features

df_train, df_test, selected_features = prepare_letor_splits('set2.train.txt')

In [ ]:
def random_ndcg_baseline(df, k=10, trials=100):
    """
    Вычисляем среднее значение NDCG для случайного ранжирования.
    Ожидаемое значение метрики обычно находится в диапазоне 0.5.

    Аргументы:
        df (pd.DataFrame): Датафрейм с данными. Должен содержать столбцы 'qid' (идентификатор запроса) 
                           и 'label' (релевантность документа).
        k (int): Параметр срезки (k) для метрики NDCG@k. По умолчанию 10.
        trials (int): Количество случайных перестановок для усреднения метрики внутри каждого запроса. 
                      По умолчанию 100.

    Возвращает:
        float: Среднее значение NDCG по всем запросам и испытаниям.
    """
    rng = np.random.RandomState(0)
    ndcgs = []
    
    for qid, group in df.groupby('qid'):
        labels = group['label'].values.reshape(1, -1)
        if labels.max() == labels.min():
            continue
            
        for _ in range(trials):
            preds = rng.rand(1, len(labels[0]))
            ndcgs.append(ndcg_score(labels, preds, k=k))
    
    return np.mean(ndcgs)


random_ndcg = random_ndcg_baseline(df_test, k=5)
print(f"✅ Рандомный baseline NDCG@10: {random_ndcg:.4f}")

#### PointWise датасет и обучение



In [ ]:
class PointwiseLTRDataset(Dataset):
    """
    Dataset для Pointwise ранжированию.
    
    Рассматривает каждый документ как независимый пример, игнорируя порядок.
    """
    def __init__(self, df, feature_cols, label_col='label'):
        # CODE HERE
    
    def __len__(self):
        # CODE HERE
    
    def __getitem__(self, idx):
        # CODE HERE

total_features = 100
feature_cols = [f'f{i}' for i in range(1, total_features+1)]

train_ds = PointwiseLTRDataset(df_train, feature_cols)
test_ds = PointwiseLTRDataset(df_test, feature_cols)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

print(f"Train docs: {len(train_ds)} | Test docs: {len(test_ds)}")
print(f"Train queries: {df_train.qid.nunique()} | Test queries: {df_test.qid.nunique()}")

In [ ]:
X_example, y_example, qids_example = list(train_loader)[0]

X_example.shape, y_example.shape, qids_example.shape

In [ ]:
class PointwiseRanker(nn.Module):
    def __init__(self, n_features=5, hidden_dim=16):
        super().__init__()
        # CODE HERE
    
    def forward(self, x):
        # CODE HERE  # Shape: (batch,)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pointwise_model = PointwiseRanker(n_features=total_features, hidden_dim=256).to(device)
optimizer = optim.Adam(pointwise_model.parameters(), lr=0.015)
criterion = nn.MSELoss()  # Pointwise regression loss

print(f"\n✅ Model: {pointwise_model}")
print(f"   Device: {device}")

In [ ]:
def train_pointwise(model, loader, optimizer, criterion, device, epochs=50):
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for X_batch, y_batch, _ in loader:  
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # CODE HERE
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1:3d} | MSE Loss: {avg_loss:.4f}")
    
    return losses

print("\n🚀 Обучаем Pointwise ...")
losses = train_pointwise(pointwise_model, train_loader, optimizer, criterion, device, epochs=300)

#### Оценка модели
Для оценки качества мы всегда используем датасет в формате *pointwise (одиночные документы), даже если модель обучалась *парно* (pairwise).


In [ ]:
def evaluate_ndcg(model, loader, device, k=4):
    """
    Вычисляет NDCG@k для каждого запроса и возвращает среднее значение по всем запросам.

    Аргументы:
        model: Обученная модель.
        loader: DataLoader с тестовыми данными.
        device: CPU или CUDA.
        k (int): Количество топовых документовдля оценки.

    Возвращает:
        tuple: (mean_ndcg, std_ndcg) — среднее значение и стандартное отклонение NDCG.
    """
    model.eval()
    preds_all, labels_all, qids_all = [], [], []
    
    with torch.no_grad():
        for X_batch, y_batch, qid_batch in loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu()
            
            preds_all.append(preds)
            labels_all.append(y_batch)
            qids_all.append(qid_batch)
    
    # Concatenate
    preds = torch.cat(preds_all).numpy()
    labels = torch.cat(labels_all).numpy()
    qids = torch.cat(qids_all).numpy()
    
    # Compute NDCG per query
    ndcgs = []
    for qid in np.unique(qids):
        mask = qids == qid
        q_labels = labels[mask].reshape(1, -1)
        q_preds = preds[mask].reshape(1, -1)
        
        # Skip if all labels identical (NDCG undefined)
        if q_labels.max() == q_labels.min():
            continue
            
        ndcgs.append(ndcg_score(q_labels, q_preds, k=k))
    
    return np.mean(ndcgs), np.std(ndcgs)

# Evaluate
ndcg_mean, ndcg_std = evaluate_ndcg(pointwise_model, test_loader, device, k=5)

print(f"Средний NDCG@4: {ndcg_mean:.4f} ± {ndcg_std:.4f}")

#### Pairwise датасет и обучение

In [ ]:
class PairwiseLTRDataset(Dataset):
    """
    Попарный (Pairwise) датасет с предвычисленными тензорами для максимальной скорости __getitem__.
    """
    def __init__(self, df, feature_cols, label_col='label', qid_col='qid', 
                 max_pairs_per_query=None, random_state=42):
        """
        Args:
            df: DataFrame со столбцами ['qid', 'label', 'f1', ..., 'fN']
            feature_cols: Список названий столбцов признаков (например, ['f1', ..., 'f100'])
            label_col/qid_col: Названия столбцов
            max_pairs_per_query: Ограничение числа пар на запрос для контроля памяти (рекомендуется: 100-200)
            random_state: Для воспроизводимости выбора пар
        """
        self.rng = np.random.RandomState(random_state)
        n_features = len(feature_cols)
        
        X_docs = torch.tensor(df[feature_cols].values, dtype=torch.float32)  
        y_docs = torch.tensor(df[label_col].values, dtype=torch.float32)      
        qids_docs = torch.tensor(df[qid_col].values, dtype=torch.long)        
        n_docs = len(df)
        
        query_to_docs = defaultdict(list)
        for idx, qid in enumerate(qids_docs.numpy()):
            query_to_docs[qid].append(idx)
        
        pair_indices = [] 
        
        for qid, doc_indices in query_to_docs.items():
            labels = y_docs[doc_indices].numpy()
            pairs = []
            
            # Генерация валидных пар предпочтений (i > j)
            for i_pos, i_idx in enumerate(doc_indices):
                for j_pos, j_idx in enumerate(doc_indices):
                    if i_pos == j_pos:
                        continue
                    if labels[i_pos] > labels[j_pos]:
                        pairs.append((i_idx, j_idx, qid))
            
            if max_pairs_per_query and len(pairs) > max_pairs_per_query:
                pairs = self.rng.choice(pairs, size=max_pairs_per_query, replace=False).tolist()
            
            pair_indices.extend(pairs)
        
        if not pair_indices:
            raise ValueError("Датасет предпочтений не сформировался, проверьте распределение предпочтений. ")
        
        n_pairs = len(pair_indices)
        
        X_pairs = torch.empty((n_pairs, 2 * n_features), dtype=torch.float32)
        y_pairs = torch.ones((n_pairs,), dtype=torch.float32)  # Тут все 1
        qid_pairs = torch.empty((n_pairs,), dtype=torch.long)
        
        for p_idx, (idx_i, idx_j, qid) in enumerate(pair_indices):
            X_pairs[p_idx, :n_features] = X_docs[idx_i]
            X_pairs[p_idx, n_features:] = X_docs[idx_j]
            qid_pairs[p_idx] = qid
        
        self.X_pairs = X_pairs      # [n_pairs, 2 * n_features]
        self.y_pairs = y_pairs      # [n_pairs]
        self.qid_pairs = qid_pairs  # [n_pairs]
        self.n_docs = n_docs        
        self.n_pairs = n_pairs
    
    def __len__(self):
        return self.n_docs
    
    def __getitem__(self, idx):
        pair_idx = idx % self.n_pairs  # Deterministic cycling
        return self.X_pairs[pair_idx], self.y_pairs[pair_idx], self.qid_pairs[pair_idx]

feature_cols = [f'f{i}' for i in range(1, 101)]
pairwise_train = PairwiseLTRDataset(df_train, feature_cols, random_state=42)

In [ ]:
def train_pairwise(model, loader, optimizer, criterion, device, epochs=50):
    """
    Pairwise training with IDENTICAL epoch semantics to train_pointwise():
      → 1 epoch = same # batches as pointwise training
      → Total training time per epoch ≈ pointwise (just 2x forward passes per batch)
    
    Args:
        criterion: MUST be nn.MarginRankingLoss(margin=...)
    """
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for X_batch, y_batch, _ in loader: 
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # CODE HERE 
            # score_i > score_j + margin
            
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1:3d} | Pairwise Loss: {avg_loss:.4f}")
    
    return losses
    
criterion = nn.MarginRankingLoss(margin=1.0)
pairwise_model = PointwiseRanker(n_features=total_features, hidden_dim=256).to(device)
train_loader = DataLoader(pairwise_train, batch_size=512, shuffle=True)
print("\n🚀 Обучение Pairwise ...")
losses = train_pairwise(
    model=pairwise_model,
    loader=train_loader,          
    optimizer=optim.Adam(pairwise_model.parameters(), lr=0.015),
    criterion=criterion,          
    device=device,
    epochs=300
)

In [ ]:
pairwise_test = PointwiseLTRDataset(df_test, feature_cols)
test_loader = DataLoader(pairwise_test, batch_size=512, shuffle=True)

ndcg_mean, ndcg_std = evaluate_ndcg(pairwise_model, test_loader, device, k=5)

print(f"Mean NDCG@4: {ndcg_mean:.4f} ± {ndcg_std:.4f}")